<a href="https://colab.research.google.com/github/dramaqueenvee/RLHF-lexical-overuse-in-Spanish-a-pilot-study/blob/main/gemma_base_nlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U transformers

GEMMA BASE MODEL

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "google/gemma-2-2b"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

print("Model loaded successfully!")

config.json:   0%|          | 0.00/818 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

Model loaded successfully!


In [ ]:
#uploading the csv and using only first half of abstracts
import pandas as pd

# Load the CSV
df = pd.read_csv("spanish_abstracts.csv")
print(f"Total abstracts: {len(df)}")
print("\nExample first half:")
print(df.iloc[0]["first_half"])

Total abstracts: 114

Example first half:
Objetivo. Estudiar la asociación de la inseguridad alimentaria con los cambios percibidos en la alimentación durante el confinamiento por Covid-19 en México. Material y métodos. El nivel de inseguridad alimentaria se obtuvo utilizando la Escala Latinoamericana y Caribeña de Seguridad Alimentaria (ELCSA) en 9 933 hogares de la Encuesta Nacional de Salud y Nutrición Continua 2020 Covid-19 (Ensanut -Continua- 2020 Covid-19). Los cambios en el consumo de grupos de alimentos durante el confinamiento se clasificaron en negativos, positivos o


In [ ]:
# prompting the model to continue the abstracts
base_continuations = []

for i, row in df.iterrows():
    prompt = f'Continúa el siguiente resumen científico: "{row["first_half"]}"'

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,  # roughly same length as second half
            do_sample=False      # deterministic output
        )

    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Remove the prompt from the output
    continuation = generated[len(prompt):]

    base_continuations.append({
        "pmid": row["pmid"],
        "continuation": continuation.strip()
    })

    if i % 10 == 0:
        print(f"Processed {i}/{len(df)}")

# Save results
base_df = pd.DataFrame(base_continuations)
base_df.to_csv("base_continuations.csv", index=False)
print("Done! Saved base_continuations.csv")

Processed 0/114
Processed 10/114
Processed 20/114
Processed 30/114
Processed 40/114
Processed 50/114
Processed 60/114
Processed 70/114
Processed 80/114
Processed 90/114
Processed 100/114
Processed 110/114
Done! Saved base_continuations.csv


In [ ]:
import pandas as pd
import re

df = pd.read_csv("base_continuations.csv")

print(f"Total rows: {len(df)}")
print(f"Empty continuations: {df['continuation'].isna().sum()}")

def clean_continuation(text):
    if pd.isna(text):
        return ""
    # Remove HTML tags
    text = re.sub(r'<[^>]+>', '', text)
    # Fix escaped quotes
    text = text.replace('""', '"')
    # Clean extra whitespace
    text = " ".join(text.split())
    return text.strip()

df['continuation'] = df['continuation'].apply(clean_continuation)

# Remove rows with empty continuations
df_clean = df[df['continuation'] != ""].reset_index(drop=True)

print(f"After removing empty: {len(df_clean)} continuations")

df_clean.to_csv("base_continuations_clean.csv", index=False, encoding="utf-8-sig")
print("Saved to base_continuations_clean.csv")

# Quick check
print("\nExample")
print(df_clean.iloc[1]["continuation"])

Total rows: 114
Empty continuations: 65
After removing empty: 49 continuations
Saved to base_continuations_clean.csv

--- Example ---
La palabra que falta en este fragmento es enfermos .
